# 02 - YOLO detector

One-stage detector trained on the fused targets (all radiologist boxes, including single-radiologist boxes).

**Inputs:** output of `00_prep` + `vbd_utils`. **GPU and internet required** (pip install + pretrained weights).

**Outputs:** `oof_yolo_fold{k}.csv`, `test_yolo_fold{k}.csv` (normalized boxes, score >= 0.001), weights in `yolo_runs/`.

**Rules from the EDA:** no horizontal flip, no mosaic, no mixup, no copy-paste (anatomy); 1024 input; class-aware NMS (overlapping boxes of different classes must survive).

**Workflow:** `FOLDS = [0]` first. A fold at 1024 can take several hours on a T4, so plan one or two folds per session.

In [ ]:
# Import the shared utilities (upload vbd_utils.py as a Kaggle Dataset and attach it)
import os, sys, glob, warnings
def _locate(name, root='/kaggle/input'):
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('png1024', 'images', 'labels', 'train', 'test')]
        if name in files:
            return cur
    raise FileNotFoundError(f'{name} not found - attach the vbd-utils dataset')
sys.path.insert(0, _locate('vbd_utils.py'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import vbd_utils as U
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
OUT = '/kaggle/working'

In [ ]:
!pip install -q ultralytics ensemble-boxes
# ---------------- configuration ----------------
FOLDS = [0]
WEIGHTS = 'yolov8m.pt'      # any Ultralytics detection checkpoint of similar size
EPOCHS = 40
BATCH = 8                   # lower to 4 if CUDA runs out of memory
PATIENCE = 10

In [ ]:
from ultralytics import YOLO
from tqdm.auto import tqdm

PNG_DIR = U.find_dir('png1024')
meta = pd.read_csv(U.find_file('meta.csv'))
folds = pd.read_csv(U.find_file('folds.csv'), index_col=0).fold
targets = pd.read_csv(U.find_file('targets_fused.csv'))
test_ids = meta.loc[meta.split == 'test', 'image_id'].tolist()
YDIR = f'{OUT}/yolo'

## Build the YOLO dataset
Ultralytics finds a label file by replacing `/images/` with `/labels/` in the image path. Images are symlinked (no copy); normal images get an empty label file (background).

In [ ]:
os.makedirs(f'{YDIR}/images', exist_ok=True); os.makedirs(f'{YDIR}/labels', exist_ok=True)
by_img = targets.groupby('image_id')
for iid in tqdm(folds.index):
    dst = f'{YDIR}/images/{iid}.png'
    if not os.path.exists(dst):
        os.symlink(f'{PNG_DIR}/{iid}.png', dst)
    lines = []
    if iid in by_img.groups:
        for r in by_img.get_group(iid).itertuples():
            lines.append(f'{r.class_id} {(r.x_min + r.x_max) / 2:.6f} {(r.y_min + r.y_max) / 2:.6f} '
                         f'{r.x_max - r.x_min:.6f} {r.y_max - r.y_min:.6f}')
    with open(f'{YDIR}/labels/{iid}.txt', 'w') as f:
        f.write('\n'.join(lines))

names = '[' + ', '.join(f"'{n}'" for n in U.CLASSES) + ']'
for k in range(5):
    train_list = open(U.find_file(f'det_train_fold{k}.txt')).read().split()
    val_list = folds.index[folds == k]
    with open(f'{YDIR}/train_fold{k}.txt', 'w') as f:
        f.write('\n'.join(f'{YDIR}/images/{i}.png' for i in train_list))
    with open(f'{YDIR}/val_fold{k}.txt', 'w') as f:
        f.write('\n'.join(f'{YDIR}/images/{i}.png' for i in val_list))
    with open(f'{YDIR}/data_fold{k}.yaml', 'w') as f:
        f.write(f'train: {YDIR}/train_fold{k}.txt\nval: {YDIR}/val_fold{k}.txt\nnc: 14\nnames: {names}\n')
print(open(f'{YDIR}/data_fold0.yaml').read())

## Train
Ultralytics prints the full list of hyper-parameters at the start of training. **Check that `fliplr=0.0` and `mosaic=0.0`** - defaults differ between versions.

Its built-in validation (mAP50, mAP50-95) is only a progress signal. Our metric is computed below with the competition rules (IoU 0.4, VOC AP).

In [ ]:
def to_df(results):
    rows = []
    for r in results:
        iid = os.path.basename(r.path)[:-4]
        b = r.boxes
        for (x1, y1, x2, y2), c, s in zip(b.xyxyn.tolist(), b.cls.tolist(), b.conf.tolist()):
            rows.append((iid, int(c), s, x1, y1, x2, y2))
    return pd.DataFrame(rows, columns=U.PRED_COLS)

def predict(model, paths, bs=16):
    out = []
    for i in tqdm(range(0, len(paths), bs)):
        out.append(to_df(model.predict(paths[i:i + bs], imgsz=U.IMG_SIZE, conf=0.001, iou=0.5,
                                       agnostic_nms=False, max_det=300, verbose=False)))
    return pd.concat(out, ignore_index=True)

for k in FOLDS:
    model = YOLO(WEIGHTS)
    model.train(data=f'{YDIR}/data_fold{k}.yaml', imgsz=U.IMG_SIZE, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                seed=42, cos_lr=True, workers=4, project=f'{OUT}/yolo_runs', name=f'fold{k}', exist_ok=True,
                fliplr=0.0, flipud=0.0, mosaic=0.0, mixup=0.0, copy_paste=0.0,      # anatomy-breaking augs OFF
                degrees=5, translate=0.1, scale=0.2, hsv_h=0.0, hsv_s=0.0, hsv_v=0.2)
    best = YOLO(f'{OUT}/yolo_runs/fold{k}/weights/best.pt')
    predict(best, [f'{YDIR}/images/{i}.png' for i in folds.index[folds == k]]).to_csv(f'{OUT}/oof_yolo_fold{k}.csv', index=False)
    predict(best, [f'{PNG_DIR}/{i}.png' for i in test_ids]).to_csv(f'{OUT}/test_yolo_fold{k}.csv', index=False)

In [ ]:
# Training curves produced by Ultralytics
from IPython.display import Image as IImage, display
for k in FOLDS:
    p = f'{OUT}/yolo_runs/fold{k}/results.png'
    if os.path.exists(p):
        display(IImage(p, width=1100))

## Evaluation report
Uses every `oof_yolo_fold*.csv` available. A detector does not predict No finding, so compare detectors on **mAP over the 14 findings**; the full 15-class metric is measured after the soft combine (notebook 04).

Contents: AP per class, precision-recall curves, detection confusion matrix, recall by box size and by radiologist agreement, and example images (worst / best / random).

In [ ]:
oof = U.load_concat('oof_yolo_fold*.csv')
eval_ids = folds.index[folds.isin(folds.loc[oof.image_id.unique()].unique())]
gt = U.build_gt(targets, eval_ids)
print('images evaluated:', len(eval_ids))
res = U.full_detection_report(gt, oof, PNG_DIR, 'YOLO (out-of-fold)', score_thr=0.2)

In [ ]:
# How the score threshold trades recall for false positives (useful when reading the confusion matrix)
for t in [0.05, 0.1, 0.2, 0.3, 0.5]:
    g = res['gt'][res['gt'].class_id != U.NO_FINDING]
    print(f'score >= {t:.2f}: recall {(g.det_score >= t).mean():.3f} | predictions per image {(oof.score >= t).sum() / len(eval_ids):.2f}')